# 📦 Retail Demand Forecasting — End-to-End ML Pipeline

> **Dataset**: Rossmann Store Sales (synthetic mirror)  
> **Models**: SARIMA + Facebook Prophet  
> **KPIs**: MAPE ≤ 15% | Inventory Cost Reduction ≥ 10%

---
### Pipeline Overview
```
Raw Data → Cleaning → EDA → Stationarity → Modeling → Evaluation → Inventory Recs
```

In [ ]:
# ── Environment setup ─────────────────────────────────────────────
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'src')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

OUTPUT = Path('outputs')
OUTPUT.mkdir(exist_ok=True)

print('✅ Libraries loaded')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')

---
## 1. Business Context & KPI Definition

In [ ]:
BUSINESS_CONTEXT = {
    'Company'          : 'RetailCo — Mid-size retail chain (500 stores)',
    'Problem'          : 'Demand fluctuation → 12–18% inventory waste annually',
    'Forecast Horizon' : '6 weeks ahead, weekly granularity',
    'KPI_1'            : 'MAPE ≤ 15%  (primary accuracy target)',
    'KPI_2'            : 'Inventory Cost Reduction ≥ 10%',
    'KPI_3'            : 'Stockout Rate Reduction ≥ 40%',
    'Dataset'          : 'Rossmann Store Sales — https://kaggle.com/competitions/rossmann-store-sales',
}

print('=' * 55)
print('  BUSINESS CONTEXT')
print('=' * 55)
for k, v in BUSINESS_CONTEXT.items():
    print(f'  {k:<22}: {v}')
print('=' * 55)

---
## 2. Data Preparation

In [ ]:
from data_prep import (
    generate_rossmann_sample,
    clean_and_engineer,
    resample_store,
    train_test_split_ts,
)

# Generate / load data
DATA_PATH = Path('data/rossmann_clean.csv')
if DATA_PATH.exists():
    df_clean = pd.read_csv(DATA_PATH, parse_dates=['Date'])
    print(f'Loaded existing clean data: {len(df_clean):,} rows')
else:
    print('Generating synthetic Rossmann dataset …')
    raw      = generate_rossmann_sample(n_stores=5, start='2020-01-01', end='2023-12-31')
    raw.to_csv('data/rossmann_sample.csv', index=False)
    print('\nCleaning & feature engineering …')
    df_clean = clean_and_engineer(raw)
    df_clean.to_csv(DATA_PATH, index=False)

df_clean.head()

In [ ]:
print('Dataset shape   :', df_clean.shape)
print('Date range      :', df_clean['Date'].min().date(), '→', df_clean['Date'].max().date())
print('Stores          :', df_clean['Store'].nunique())
print('Missing values  :', df_clean.isnull().sum().sum())
print()
df_clean.describe().round(1)

In [ ]:
# Resample to weekly & split
STORE_ID    = 1
TEST_WEEKS  = 12

ts          = resample_store(df_clean, store_id=STORE_ID, freq='W')
train, test = train_test_split_ts(ts, test_periods=TEST_WEEKS)

print(f'\nWeekly series: {len(ts)} weeks')
ts.tail()

---
## 3. Exploratory Data Analysis

In [ ]:
from eda import (
    plot_sales_trend,
    plot_seasonality,
    plot_decomposition,
    stationarity_report,
    plot_acf_pacf,
    plot_store_heatmap,
)

print('[1] Sales trend …')
plot_sales_trend(df_clean, store_id=STORE_ID)

print('[2] Seasonality patterns …')
plot_seasonality(df_clean, store_id=STORE_ID)

print('[3] Seasonal decomposition …')
plot_decomposition(ts)

In [ ]:
print('[4] Stationarity tests …')
stats = stationarity_report(ts['Sales'], name='Weekly Sales (Store 1)')

In [ ]:
print('[5] ACF / PACF (for ARIMA order selection) …')
series_for_acf = ts['Sales'].diff().dropna() if not stats['adf_stationary'] else ts['Sales']
plot_acf_pacf(series_for_acf, lags=40)

print('[6] Store comparison heatmap …')
plot_store_heatmap(df_clean)

### EDA Summary
| Finding | Implication |
|---|---|
| Clear weekly seasonality (Mon/Fri peaks) | Include seasonal component in model |
| Annual cycle: Nov–Dec surge | Seasonal differencing D=1 |
| ADF → likely non-stationary | Use d=1 differencing in ARIMA |
| ACF slow decay | AR term needed (p ≥ 1) |
| PACF cuts off at lag 1 | p=1 sufficient |

---
## 4. Modeling

### 4a. SARIMA

In [ ]:
from modeling import SARIMAForecaster

sarima = SARIMAForecaster(order=(1,1,1), seasonal_order=(1,1,1,52))
sarima.fit(train)
sarima_metrics = sarima.evaluate(test)

# Align predictions to test index
pred_sarima       = sarima._pred_mean.copy()
pred_sarima.index = test.index

In [ ]:
sarima.plot_forecast(test, title='SARIMA(1,1,1)(1,1,1)[52] — 12-Week Forecast')
sarima.plot_diagnostics()
print('Plots saved to outputs/')

### 4b. Prophet

In [ ]:
from modeling import ProphetForecaster, PROPHET_AVAILABLE

all_metrics  = [sarima_metrics]
predictions  = {'SARIMA': pred_sarima}

if PROPHET_AVAILABLE:
    prophet = ProphetForecaster(changepoint_prior_scale=0.15)
    prophet.fit(train)
    prophet_metrics = prophet.evaluate(test)

    fdf          = prophet._forecast_df.tail(len(test))
    pred_prophet = pd.Series(fdf['yhat'].values, index=test.index)

    prophet.plot_forecast(test)
    prophet.plot_components()

    all_metrics.append(prophet_metrics)
    predictions['Prophet'] = pred_prophet
    print('Prophet complete.')
else:
    print('Prophet not installed — pip install prophet')

In [ ]:
from modeling import compare_models, plot_model_comparison

metrics_df = compare_models(all_metrics)
plot_model_comparison(
    sarima_metrics,
    prophet_metrics if PROPHET_AVAILABLE else None
)

---
## 5. Evaluation

In [ ]:
from evaluate import (
    plot_actual_vs_predicted,
    plot_residuals,
    metrics_dashboard,
    inventory_recommendations,
    plot_inventory_plan,
    print_business_insights,
)

# Residual std for safety stock calculation
residuals = test['Sales'].values - np.maximum(pred_sarima.values, 0)
resid_std  = float(np.std(residuals))

print('[1] Actual vs Predicted …')
plot_actual_vs_predicted(test, predictions)

print('[2] Residual analysis …')
plot_residuals(test, pred_sarima, model_name='SARIMA')

In [ ]:
print('[3] Metrics dashboard …')
metrics_dashboard(all_metrics)

---
## 6. Inventory Recommendations

In [ ]:
recs = inventory_recommendations(
    forecast=pred_sarima,
    residual_std=resid_std,
    unit_cost=5.0,
    holding_cost_pct=0.20,
    stockout_cost_pct=0.40,
    service_level_z=1.645,   # 95% service level
)

plot_inventory_plan(recs)
recs

---
## 7. Business Insights

In [ ]:
print_business_insights(all_metrics)

---
## 8. Output Summary

In [ ]:
print('\n📁 Generated Outputs:')
for f in sorted(OUTPUT.glob('*')):
    size = f.stat().st_size
    print(f'   {f.name:<35} {size/1024:.1f} KB')

print('\n✅ Pipeline complete.')
print(f'   Best MAPE  : {min(all_metrics, key=lambda m: m["MAPE"])["MAPE"]:.2f}%')
print(f'   KPI Status : {"✅ MET" if min(all_metrics, key=lambda m: m["MAPE"])["MAPE"] <= 15 else "❌ MISSED"}')